Assignment 2 – Scraping, APIs y cruce por ubigeo    
    Parte 1------Temporada analizada: 1 de enero al 31 de mayo de 2022.

#1.Revisión

Antes de realizar el scraping, se revisa el archivo robots.txt de gob.pe para identificar las restricciones establecidas.

In [55]:
import requests

url_robots = "https://www.gob.pe/robots.txt"

headers = {
    "User-Agent": "Mozilla/5.0"
}

respuesta = requests.get(url_robots, headers=headers)

print("Código de estado:", respuesta.status_code)
print(respuesta.text)

Código de estado: 200
# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



#Interpretación

A partir de la revisión del archivo `robots.txt` de gob.pe se puede observar que existen determinadas restricciones para el acceso automatizado, principalmente para rutas como `/admin/` y algunas direcciones que utilizan el parámetro `sheet`;sin embargo, la ruta `/busquedas`, que será utilizada para obtener la información, no se encuentran dentro de estas restricciones.

#2. Prueba de Selenium con un mes

In [56]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [57]:
url_enero = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas&"
    "institucion[]=pcm&"
    "term=estado%20de%20emergencia%20precipitaciones&"
    "sort_by=recent&"
    "desde=01-01-2022&"
    "hasta=31-01-2022"
)

print(url_enero)

https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2022&hasta=31-01-2022


In [58]:
driver = webdriver.Chrome()

driver.get(url_enero)

wait = WebDriverWait(driver, 20)

articles = wait.until(
    EC.presence_of_all_elements_located((By.TAG_NAME, "article"))
)

print("Cantidad de artículos encontrados:", len(articles))

Cantidad de artículos encontrados: 8


#3. Número total de resultados

In [59]:
texto_pagina = driver.find_element(
    By.TAG_NAME, "body"
).text

lineas = texto_pagina.split("\n")

lineas_resultados = []

for linea in lineas:
    if "Resultados" in linea:
        lineas_resultados.append(linea)

print(lineas_resultados)

['8 Resultados']


In [60]:
texto_resultados = lineas_resultados[0]

partes = texto_resultados.split()

resultados_totales = int(partes[0])

print("Resultados totales:", resultados_totales)

Resultados totales: 8


#4. Extracción de información de resultados

In [61]:
datos_enero = []

for article in articles:
    
    enlaces = article.find_elements(By.TAG_NAME, "a")
    
    numero = enlaces[0].text.strip()
    enlace = enlaces[0].get_attribute("href")
    
    lineas = article.text.split("\n")
    
    fecha = None
    titulo = None
    
    for i in range(len(lineas)):
        
        if lineas[i].startswith("Publicado:"):
            fecha = lineas[i].replace("Publicado:", "").strip()
            
            if i + 1 < len(lineas):
                titulo = lineas[i + 1].strip()
    
    registro = {
        "numero": numero,
        "fecha": fecha,
        "titulo": titulo,
        "enlace": enlace
    }
    
    datos_enero.append(registro)

In [62]:
import pandas as pd

df_enero = pd.DataFrame(datos_enero)

df_enero

,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 011-2022-PCM,30 de enero de 2022,Decreto Supremo que modifica el numeral 14.2 d...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 010-2022-PCM,29 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 009-2022-PCM,28 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 004-2022-PCM,25 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 007-2022-PCM,18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 006-2022-PCM,18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 005-2022-PCM,16 de enero de 2022,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...
7,Decreto Supremo N.° 002-2022-PCM,6 de enero de 2022,Decreto Supremo que modifica el numeral 8.1 de...,https://www.gob.pe/institucion/pcm/normas-lega...


#5. Extracción total

In [63]:
import time

meses = [
    ("enero", "01-01-2022", "31-01-2022"),
    ("febrero", "01-02-2022", "28-02-2022"),
    ("marzo", "01-03-2022", "31-03-2022"),
    ("abril", "01-04-2022", "30-04-2022"),
    ("mayo", "01-05-2022", "31-05-2022")
]

In [64]:
todos_datos = []
verificacion = []

for mes, desde, hasta in meses:

    url = (
        "https://www.gob.pe/busquedas?"
        "contenido[]=normas&"
        "institucion[]=pcm&"
        "term=estado%20de%20emergencia%20precipitaciones&"
        "sort_by=recent&"
        f"desde={desde}&"
        f"hasta={hasta}"
    )

    driver.get(url)

    articles = WebDriverWait(driver, 20).until(
        EC.presence_of_all_elements_located(
            (By.TAG_NAME, "article")
        )
    )

    texto_pagina = driver.find_element(
        By.TAG_NAME, "body"
    ).text

    lineas_pagina = texto_pagina.split("\n")

    resultados_totales = 0

    for linea in lineas_pagina:
        if "Resultados" in linea:
            partes = linea.split()
            resultados_totales = int(partes[0])
            break

    datos_mes = []

    for article in articles:

        enlaces = article.find_elements(By.TAG_NAME, "a")

        numero = enlaces[0].text.strip()
        enlace = enlaces[0].get_attribute("href")

        lineas = article.text.split("\n")

        fecha = None
        titulo = None

        for i in range(len(lineas)):

            if lineas[i].startswith("Publicado:"):

                fecha = lineas[i].replace(
                    "Publicado:", ""
                ).strip()

                if i + 1 < len(lineas):
                    titulo = lineas[i + 1].strip()

        registro = {
            "mes": mes,
            "numero": numero,
            "fecha": fecha,
            "titulo": titulo,
            "enlace": enlace
        }

        datos_mes.append(registro)
        todos_datos.append(registro)

    resumen_mes = {
        "mes": mes,
        "resultados_totales": resultados_totales,
        "resultados_extraidos": len(datos_mes)
    }

    verificacion.append(resumen_mes)

    print(
        mes,
        "- Totales:", resultados_totales,
        "- Extraídos:", len(datos_mes)
    )

    time.sleep(2)

enero - Totales: 8 - Extraídos: 8
febrero - Totales: 5 - Extraídos: 5
marzo - Totales: 5 - Extraídos: 5
abril - Totales: 4 - Extraídos: 4
mayo - Totales: 7 - Extraídos: 7


In [65]:
df_decretos = pd.DataFrame(todos_datos)

print("Total de registros:", len(df_decretos))

df_decretos.head()

Total de registros: 29


,mes,numero,fecha,titulo,enlace
0,enero,Decreto Supremo N.° 011-2022-PCM,30 de enero de 2022,Decreto Supremo que modifica el numeral 14.2 d...,https://www.gob.pe/institucion/pcm/normas-lega...
1,enero,Decreto Supremo N.° 010-2022-PCM,29 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,enero,Decreto Supremo N.° 009-2022-PCM,28 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,enero,Decreto Supremo N.° 004-2022-PCM,25 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,enero,Decreto Supremo N.° 007-2022-PCM,18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


#6. Verificación de resultados por mes

In [66]:
df_verificacion = pd.DataFrame(verificacion)

df_verificacion

,mes,resultados_totales,resultados_extraidos
0,enero,8,8
1,febrero,5,5
2,marzo,5,5
3,abril,4,4
4,mayo,7,7


#7. Eliminación de normas repetidas

In [67]:
duplicados = df_decretos.duplicated(
    subset="enlace"
).sum()

print("Total de registros:", len(df_decretos))
print("Normas duplicadas:", duplicados)

Total de registros: 29
Normas duplicadas: 0


In [68]:
df_decretos = df_decretos.drop_duplicates(
    subset="enlace"
)

df_decretos = df_decretos.reset_index(
    drop=True
)

print(
    "Registros después de eliminar duplicados:",
    len(df_decretos)
)

Registros después de eliminar duplicados: 29
